# Evolving Ising: A Step-by-Step Tour

This notebook walks through the `evolving_ising` package from first principles:

- **Part 1** — `IsingModel`: the spin grid and neighbour graph
- **Part 2** — Spins & Metropolis: thermal equilibration
- **Part 3** — Temperature Diffusion: heat flow over bonds
- **Part 4** — Coupled Dynamics: interleaved diffusion + Metropolis
- **Part 5** — Parameter Space & Fitness Objectives
- **Part 6** — CMA-ES Optimization: learning J to channel heat
- **Part 7** — Results: animations and connectivity analysis


In [ ]:
import jax
import jax.numpy as jnp
import matplotlib
matplotlib.rcParams['animation.html'] = 'jshtml'
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import numpy as np

from evolving_ising import IsingModel, TemperatureDiffuser, SeparableCMAES
from evolving_ising.objectives import PhysicsSetup, make_eval_fn, vec_to_Jnk, EXPERIMENTS

print("JAX devices:", jax.devices())


## Part 1 — `IsingModel`: The Grid

`IsingModel` represents a 2D rectangular lattice of Ising spins s ∈ {−1, +1}.
At construction it precomputes:

- `neighbors (N, K)` — flat index of each site's K neighbours
- `mask (N, K)` — True where the slot is a real (valid) neighbour
- `rev_slot (N, K)` — reverse-direction slot index (used for symmetric bonds)
- Colour masks for the checkerboard Metropolis update scheme

Let's start with a tiny **4×4 open** grid so we can inspect the arrays directly.


In [ ]:
m_tiny = IsingModel((4, 4), neighborhood='von_neumann', boundary='open')
print(f"Grid 4x4   N={m_tiny.n}   K={m_tiny.K} slots per site")
print()
print("neighbors[:5] (flat index of each neighbour slot):")
print(m_tiny.neighbors[:5])
print()
print("mask[:5]  -- 1 = valid bond, 0 = boundary/invalid:")
print(m_tiny.mask[:5].astype(int))


In [ ]:
# Visualize the 4x4 neighbour graph
H_t, W_t = 4, 4
fig, ax = plt.subplots(figsize=(5, 5))
for i in range(H_t):
    for j in range(W_t):
        n = i * W_t + j
        ax.scatter(j, -i, s=300, c='steelblue', zorder=3)
        ax.text(j, -i, str(n), ha='center', va='center',
                color='white', fontsize=9, fontweight='bold', zorder=4)
        for k in range(m_tiny.K):
            if m_tiny.mask[n, k]:
                nb = int(m_tiny.neighbors[n, k])
                ni_nb, nj_nb = nb // W_t, nb % W_t
                dx, dy = nj_nb - j, -(ni_nb - i)
                ax.annotate('', xy=(j + dx*0.38, -i + dy*0.38),
                            xytext=(j - dx*0.38, -i - dy*0.38),
                            arrowprops=dict(arrowstyle='->', color='#aaa', lw=1.0))
ax.set_xlim(-0.7, W_t - 0.3)
ax.set_ylim(-H_t + 0.3, 0.7)
ax.set_title('Neighbour graph — 4×4 open von Neumann\n(numbers = flat site index)')
ax.axis('off')
plt.tight_layout()
plt.show()


## Part 2 — Spins & Metropolis

Spins are `int8` arrays of shape `(B, N)` — B independent chains, N sites.

The **checkerboard Metropolis** update partitions sites into colour classes
(2 for von Neumann, 4 for Moore) so that an entire class can be updated in
parallel without data races.  `metropolis_checkerboard_sweeps` runs one or
more full sweeps (all colour classes) and returns the updated spins plus the
energy of the final configuration.


In [ ]:
key = jax.random.PRNGKey(0)
m8  = IsingModel((8, 8), neighborhood='von_neumann', boundary='periodic_lr')
N8, K8 = m8.n, m8.K

# Uniform ferromagnetic couplings: J = 1 at all valid bonds
J_unif8 = jnp.ones((N8, K8), dtype=jnp.float32) * m8.mask

key, sub = jax.random.split(key)
spins8   = m8.init_spins(sub, batch_size=1)   # (1, N)

print(f"Spins shape: {spins8.shape}  dtype: {spins8.dtype}")
print(f"Unique values: {sorted(int(x) for x in jnp.unique(spins8))}")
print(f"Initial energy: {float(m8.energy(J_unif8, spins8)[0]):.2f}")

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(spins8[0].reshape(8, 8), cmap='gray', vmin=-1, vmax=1)
ax.set_title('Random spins (8x8)')
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Energy relaxation at high T (disordered) vs low T (ordered)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
final_spins = {}

for ax, T_val, label in zip(axes,
                             [5.0,          0.5],
                             ['High T=5.0', 'Low T=0.5']):
    key, sub_init = jax.random.split(key)
    sp = m8.init_spins(sub_init, batch_size=4)
    E_hist = [float(jnp.mean(m8.energy(J_unif8, sp)))]
    for _ in range(150):
        key, sub = jax.random.split(key)
        sp, E_b = m8.metropolis_checkerboard_sweeps(
            sub, sp, J_unif8, jnp.array(T_val), num_sweeps=1)
        E_hist.append(float(jnp.mean(E_b)))
    ax.plot(E_hist, lw=1.2)
    ax.set_title(f'{label} — energy relaxation (4 chains)')
    ax.set_xlabel('Sweep')
    ax.set_ylabel('Mean energy')
    ax.grid(True, alpha=0.3)
    final_spins[T_val] = sp

plt.suptitle('Metropolis on 8x8, uniform J=1')
plt.tight_layout()
plt.show()

# Spin snapshots
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, T_val, label in zip(axes, [5.0, 0.5], ['High T (disordered)', 'Low T (ordered)']):
    ax.imshow(final_spins[T_val][0].reshape(8, 8), cmap='gray', vmin=-1, vmax=1)
    ax.set_title(label)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Part 3 — Temperature Diffusion

`TemperatureDiffuser` propagates a temperature field T over the neighbour graph:

    T_{t+1}[i] = (1 − α) T_t[i]  +  α Σ_k  W_norm[i,k] · T_t[nbr[i,k]]

`W_norm` is a row-normalised conductance derived from J_nk (via `conductance_mode`).
Pinned boundary sites are hard-reset to fixed values after each step.

The diffuser is **stateless** — `neighbors` and `mask` come from IsingModel at call time.


In [ ]:
HOT_T, COLD_T = 3.0, 0.0

H_d, W_d, N_d = 16, 16, 16*16
m_d   = IsingModel((H_d, W_d), neighborhood='von_neumann', boundary='periodic_lr')
J_d   = jnp.ones((N_d, m_d.K), dtype=jnp.float32) * m_d.mask

diffuser = TemperatureDiffuser(alpha=0.35, conductance_mode='abs', normalize_mode='row')

cols_d       = jnp.arange(W_d, dtype=jnp.int32)
bot_d        = (H_d - 1) * W_d + cols_d
pin_mask_d   = jnp.zeros(N_d, dtype=bool).at[bot_d].set(True)
pin_values_d = jnp.zeros(N_d, dtype=jnp.float32).at[bot_d].set(HOT_T)

print(f"alpha={diffuser.alpha}  conductance_mode={diffuser._c_mode}")
print(f"Bottom row pinned at T={HOT_T}, rest starts cold.")


In [ ]:
# Step through diffusion and snapshot at steps 0, 3, 10, 30
snap_steps = [0, 3, 10, 30]
T_d = jnp.zeros((1, N_d), dtype=jnp.float32)   # (B=1, N)
snaps = {0: np.array(T_d[0])}

for step in range(1, max(snap_steps) + 1):
    T_d = diffuser.step(m_d.neighbors, J_d, m_d.mask, T_d, pin_mask_d, pin_values_d)
    if step in snap_steps:
        snaps[step] = np.array(T_d[0])

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, s in zip(axes, snap_steps):
    im = ax.imshow(snaps[s].reshape(H_d, W_d), cmap='magma', vmin=COLD_T, vmax=HOT_T)
    ax.set_title(f'Step {s}')
    ax.axis('off')
plt.suptitle('Temperature diffusion — uniform J, bottom row pinned hot', fontsize=11)
plt.colorbar(im, ax=axes[-1], fraction=0.05)
plt.tight_layout()
plt.show()


In [ ]:
# Animate the diffusion from a cold start
T_anim = jnp.zeros((1, N_d), dtype=jnp.float32)
T_anim_frames = []
for _ in range(80):
    T_anim = diffuser.step(m_d.neighbors, J_d, m_d.mask, T_anim, pin_mask_d, pin_values_d)
    T_anim_frames.append(np.array(T_anim[0]))

fig_da, ax_da = plt.subplots(figsize=(4, 4))
im_da = ax_da.imshow(T_anim_frames[0].reshape(H_d, W_d), cmap='magma', vmin=COLD_T, vmax=HOT_T)
ax_da.set_title('Temperature diffusion (16x16, uniform J)')
ax_da.axis('off')

def _upd_diff(i):
    im_da.set_data(T_anim_frames[i].reshape(H_d, W_d))
    return (im_da,)

ani_diff = animation.FuncAnimation(fig_da, _upd_diff,
                                   frames=len(T_anim_frames), interval=50, blit=True)
plt.close(fig_da)
display(HTML(ani_diff.to_jshtml()))


## Part 4 — Coupled Dynamics

The full model **interleaves** temperature diffusion with Metropolis spin updates.
Each iteration:

1. Diffuse T over the bond graph (J acts as conductance)
2. Run Metropolis sweeps using per-site temperature T[i]

Hot sites flip more freely (high entropy); cold sites freeze into aligned domains.
The feedback between T and spins is the core of the model.


In [ ]:
H_c, W_c, N_c = 16, 16, 16*16
m_c = IsingModel((H_c, W_c), neighborhood='von_neumann', boundary='periodic_lr')
J_c = jnp.ones((N_c, m_c.K), dtype=jnp.float32) * m_c.mask

cols_c       = jnp.arange(W_c, dtype=jnp.int32)
bot_c        = (H_c - 1) * W_c + cols_c
pin_mask_c   = jnp.zeros(N_c, dtype=bool).at[bot_c].set(True)
pin_values_c = jnp.zeros(N_c, dtype=jnp.float32).at[bot_c].set(HOT_T)
T0_c         = jnp.repeat(jnp.linspace(COLD_T, HOT_T, H_c, dtype=jnp.float32), W_c)

key, sub_sc = jax.random.split(key)
spins_c = m_c.init_spins(sub_sc, 1)
T_c     = T0_c[None]   # (1, N_c)

T_c_frames, S_c_frames = [], []
for _ in range(80):
    T_c = diffuser.diffuse(m_c.neighbors, J_c, m_c.mask, T_c,
                           steps=2, pin_mask=pin_mask_c, pin_values=pin_values_c)
    key, sub = jax.random.split(key)
    spins_c, _ = m_c.metropolis_checkerboard_sweeps(sub, spins_c, J_c, T_c, num_sweeps=2)
    T_c_frames.append(np.array(T_c[0]))
    S_c_frames.append(np.array(spins_c[0]))

top_row_c = float(np.mean(T_c_frames[-1][:W_c]))
print(f"Final top-row T (uniform J, 16x16): {top_row_c:.4f}  (baseline before optimization)")


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
im_tc = axs[0].imshow(T_c_frames[-1].reshape(H_c, W_c), cmap='magma', vmin=COLD_T, vmax=HOT_T)
axs[0].set_title('Temperature (final)')
axs[0].axis('off')
plt.colorbar(im_tc, ax=axs[0], fraction=0.046)
axs[1].imshow(S_c_frames[-1].reshape(H_c, W_c), cmap='gray', vmin=-1, vmax=1)
axs[1].set_title('Spins (final)')
axs[1].axis('off')
plt.suptitle('Coupled diffusion + Metropolis — 16x16, uniform J')
plt.tight_layout()
plt.show()


In [ ]:
n_frames_c = len(T_c_frames)

fig_cT, ax_cT = plt.subplots(figsize=(4, 4))
im_cT = ax_cT.imshow(T_c_frames[0].reshape(H_c, W_c), cmap='magma', vmin=COLD_T, vmax=HOT_T)
ax_cT.set_title('Temperature (uniform J)')
ax_cT.axis('off')
def _upd_cT(i):
    im_cT.set_data(T_c_frames[i].reshape(H_c, W_c))
    return (im_cT,)
ani_cT = animation.FuncAnimation(fig_cT, _upd_cT, frames=n_frames_c, interval=60, blit=True)
plt.close(fig_cT)
display(HTML(ani_cT.to_jshtml()))

fig_cS, ax_cS = plt.subplots(figsize=(4, 4))
im_cS = ax_cS.imshow(S_c_frames[0].reshape(H_c, W_c), cmap='gray', vmin=-1, vmax=1)
ax_cS.set_title('Spins (uniform J)')
ax_cS.axis('off')
def _upd_cS(i):
    im_cS.set_data(S_c_frames[i].reshape(H_c, W_c))
    return (im_cS,)
ani_cS = animation.FuncAnimation(fig_cS, _upd_cS, frames=n_frames_c, interval=60, blit=True)
plt.close(fig_cS)
display(HTML(ani_cS.to_jshtml()))


## Part 5 — Parameter Space & Fitness Objectives

The CMA-ES optimises a flat vector **θ ∈ ℝᴰ** (D = number of valid bond slots).
The mapping from parameters to couplings is:

    J_nk[flat_idx] = softplus(θ) × j_scale

`PhysicsSetup` bundles the shared physical configuration into one object.
`make_eval_fn(objective, setup)` returns JIT-compiled `(eval_single, eval_population)`.


In [ ]:
print("Available objectives:\n")
for obj_key, info in EXPERIMENTS.items():
    print(f"  {obj_key}")
    print(f"    formula : {info['formula']}")
    print(f"    desc    : {info['description'][:72]}...")
    print()


In [ ]:
# Build PhysicsSetup for a 32x32 grid
H, W = 32, 32
N    = H * W

ising        = IsingModel((H, W), neighborhood='von_neumann', boundary='periodic_lr')
diffuser_main= TemperatureDiffuser(alpha=0.35, conductance_mode='abs', normalize_mode='row')

cols       = jnp.arange(W, dtype=jnp.int32)
top_idx    = cols                          # row 0  (cold / measurement row)
bot_idx    = (H - 1) * W + cols           # row H-1 (hot boundary)
pin_mask   = jnp.zeros(N, dtype=bool).at[bot_idx].set(True)
pin_values = jnp.zeros(N, dtype=jnp.float32).at[bot_idx].set(HOT_T)
T0         = jnp.repeat(jnp.linspace(COLD_T, HOT_T, H, dtype=jnp.float32), W)
flat_idx   = jnp.where(ising.mask.reshape(-1))[0]

setup = PhysicsSetup(
    ising           = ising,
    diffuser        = diffuser_main,
    T0              = T0,
    pin_mask        = pin_mask,
    pin_values      = pin_values,
    top_idx         = top_idx,
    flat_idx        = flat_idx,
    iters_eval      = 60,
    steps_per_iter  = 2,
    sweeps_per_iter = 2,
    chains_per_eval = 8,
    j_scale         = 1.0,
)

print(f"Grid: {H}x{W}   N={N}   K={ising.K}")
print(f"Parameter dimension D = {setup.D}  (valid bond slots to optimise)")


In [ ]:
# vec_to_Jnk: theta (D,) -> J_nk (N, K)
key_d      = jax.random.PRNGKey(42)
theta_demo = jax.random.normal(key_d, (setup.D,), dtype=jnp.float32)
J_demo     = vec_to_Jnk(theta_demo, setup)

print(f"theta  shape : {theta_demo.shape}")
print(f"J_nk   shape : {J_demo.shape}")
print(f"J range      : [{float(J_demo.min()):.4f}, {float(J_demo.max()):.4f}]  (>= 0 via softplus)")
print(f"Invalid slots sum (should be 0) : {float((J_demo * ~ising.mask).sum()):.1f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(np.array(J_demo[ising.mask]), bins=50,
        color='steelblue', edgecolor='white', linewidth=0.3)
ax.set_xlabel('J value')
ax.set_ylabel('Count')
ax.set_title('Random J_nk — softplus(N(0,1)) at all valid bond slots')
plt.tight_layout()
plt.show()


In [ ]:
# Build eval functions and trigger JIT compilation
OBJECTIVE = 'max_top_temp'
eval_single, eval_population = make_eval_fn(OBJECTIVE, setup)

key_opt = jax.random.PRNGKey(0)
key_opt, sub_w = jax.random.split(key_opt)
_ = eval_single(sub_w, jnp.zeros(setup.D, dtype=jnp.float32))

print(f"Objective : '{OBJECTIVE}'")
print(f"Formula   : {EXPERIMENTS[OBJECTIVE]['formula']}")
print("eval_single compiled and ready.")


## Part 6 — CMA-ES Optimization

`SeparableCMAES` maintains a Gaussian distribution N(μ, σ² diag(C)) over θ-space
with a diagonal covariance (one variance per dimension).

Each generation:
1. `ask()` → sample population X of shape `(pop_size, D)`
2. Evaluate fitness for each candidate via `eval_population`
3. `tell(X, fitness)` → update μ, σ, C from the top-μ winners

The algorithm maximises fitness — here, the mean temperature on the top row.


In [ ]:
import time

es = SeparableCMAES(dim=setup.D, pop_size=32, sigma_init=0.5, seed=0)

CMA_ITERS  = 200
best_hist  = []
bsf_hist   = []   # best-so-far (monotone)
mean_hist  = []
best_theta = None
best_fit   = -jnp.inf

t0 = time.time()
for t in range(CMA_ITERS):
    X       = es.ask()                          # (pop_size, D)
    key_opt, sub_e = jax.random.split(key_opt)
    fitness = eval_population(sub_e, X)         # (pop_size,)
    es.tell(X, fitness)

    idx   = int(jnp.argmax(fitness))
    val   = float(fitness[idx])
    fmean = float(jnp.mean(fitness))
    if best_theta is None or val > float(best_fit):
        best_fit   = fitness[idx]
        best_theta = X[idx]

    best_hist.append(val)
    bsf_hist.append(float(best_fit))
    mean_hist.append(fmean)

    if (t + 1) % 25 == 0:
        print(f"Iter {t+1:3d}/{CMA_ITERS}  best: {float(best_fit):.4f}"
              f"  mean: {fmean:.4f}  ({time.time()-t0:.0f}s)")

print(f"\nDone in {time.time()-t0:.1f}s   best top-row T: {float(best_fit):.4f}")


In [ ]:
iters_arr = np.arange(1, CMA_ITERS + 1)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(iters_arr, bsf_hist,  lw=2.0,                   label='best so far (monotone)')
ax.plot(iters_arr, best_hist, lw=0.8, alpha=0.55,        label='best per iter (noisy)')
ax.plot(iters_arr, mean_hist, lw=0.8, linestyle='--',
        alpha=0.7, label='population mean')
ax.set_xlabel('CMA-ES iteration')
ax.set_ylabel('Top-row temperature')
ax.set_title(f'Optimization progress — {OBJECTIVE} on {H}x{W} grid')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Part 7 — Results

Recover the best coupling matrix J_nk, run a longer rollout to reach
steady state, then visualise the learned temperature field, spin configuration,
and connectivity structure.


In [ ]:
J_best = vec_to_Jnk(best_theta, setup)

key_vis = jax.random.PRNGKey(99)
key_vis, sub_sv = jax.random.split(key_vis)
spins_vis = ising.init_spins(sub_sv, 1)
T_vis     = jnp.broadcast_to(T0[None], (1, N))

T_res_frames, S_res_frames = [], []
for _ in range(120):
    T_vis = diffuser_main.diffuse(ising.neighbors, J_best, ising.mask, T_vis,
                                  steps=2, pin_mask=pin_mask, pin_values=pin_values)
    key_vis, sub_v = jax.random.split(key_vis)
    spins_vis, _  = ising.metropolis_checkerboard_sweeps(
        sub_v, spins_vis, J_best, T_vis, num_sweeps=2)
    T_res_frames.append(np.array(T_vis[0]))
    S_res_frames.append(np.array(spins_vis[0]))

top_idx_np   = np.array(top_idx)
top_row_temp = float(np.mean(T_res_frames[-1][top_idx_np]))
mean_temp    = float(np.mean(T_res_frames[-1]))
print(f"Final top-row temperature : {top_row_temp:.4f}")
print(f"Final mean temperature    : {mean_temp:.4f}")


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
im_res = axs[0].imshow(T_res_frames[-1].reshape(H, W), cmap='magma', vmin=COLD_T, vmax=HOT_T)
axs[0].set_title('Temperature — optimized J')
axs[0].axis('off')
plt.colorbar(im_res, ax=axs[0], fraction=0.046, pad=0.04)
axs[1].imshow(S_res_frames[-1].reshape(H, W), cmap='gray', vmin=-1, vmax=1)
axs[1].set_title('Spins — optimized J')
axs[1].axis('off')
plt.suptitle(f'{H}x{W}  |  top-row T = {top_row_temp:.3f}  |  mean T = {mean_temp:.3f}')
plt.tight_layout()
plt.show()


In [ ]:
# Animate temperature and spins with optimized J
n_res = len(T_res_frames)

fig_rT, ax_rT = plt.subplots(figsize=(4, 4))
im_rT = ax_rT.imshow(T_res_frames[0].reshape(H, W), cmap='magma', vmin=COLD_T, vmax=HOT_T)
ax_rT.set_title('Temperature (optimized J)')
ax_rT.axis('off')
def _upd_rT(i):
    im_rT.set_data(T_res_frames[i].reshape(H, W))
    return (im_rT,)
ani_rT = animation.FuncAnimation(fig_rT, _upd_rT, frames=n_res, interval=50, blit=True)
plt.close(fig_rT)
display(HTML(ani_rT.to_jshtml()))

fig_rS, ax_rS = plt.subplots(figsize=(4, 4))
im_rS = ax_rS.imshow(S_res_frames[0].reshape(H, W), cmap='gray', vmin=-1, vmax=1)
ax_rS.set_title('Spins (optimized J)')
ax_rS.axis('off')
def _upd_rS(i):
    im_rS.set_data(S_res_frames[i].reshape(H, W))
    return (im_rS,)
ani_rS = animation.FuncAnimation(fig_rS, _upd_rS, frames=n_res, interval=50, blit=True)
plt.close(fig_rS)
display(HTML(ani_rS.to_jshtml()))


In [ ]:
# Connectivity: in-degree (conductance received) and out-degree (conductance sent)
J_np    = np.abs(np.array(J_best))
mask_np = np.array(ising.mask)
nbr_np  = np.array(ising.neighbors)

W_conn  = J_np * mask_np.astype(np.float32)
out_deg = np.sum(W_conn, axis=1)          # (N,)
in_deg  = np.zeros(N, dtype=np.float32)
np.add.at(in_deg, nbr_np.reshape(-1), W_conn.reshape(-1))

in_norm  = in_deg  / (np.max(in_deg)  + 1e-8)
out_norm = out_deg / (np.max(out_deg) + 1e-8)
RGB = np.stack([in_norm, out_norm, np.zeros(N)], axis=-1).reshape(H, W, 3)

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(in_deg.reshape(H, W),  cmap='Reds')
axs[0].set_title('In-degree (conductance received)')
axs[0].axis('off')
axs[1].imshow(out_deg.reshape(H, W), cmap='Greens')
axs[1].set_title('Out-degree (conductance sent)')
axs[1].axis('off')
axs[2].imshow(RGB)
axs[2].set_title('R = in,  G = out')
axs[2].axis('off')
plt.suptitle('Learned connectivity structure — optimized J', fontsize=11)
plt.tight_layout()
plt.show()
